# 04 Fisher-Kolmogorov Protection Strategies


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp


def graph_laplacian(W):
    W = np.asarray(W, dtype=float)
    return np.diag(W.sum(axis=1)) - W


def protected_fk_rhs(t, y, L, kappa, alpha, protection, mode="alpha"):
    protection = np.asarray(protection, dtype=float)
    scale = 1.0 - protection

    diffusion = -kappa * (L @ y)
    growth = alpha * y * (1.0 - y)

    if mode == "alpha":
        return diffusion + scale * growth
    if mode == "kappa":
        return scale * diffusion + growth
    if mode in {"both", "kappa_alpha"}:
        return scale * (diffusion + growth)
    raise ValueError("mode must be 'alpha', 'kappa', or 'both'.")


def make_protection_vector(n_regions, protected_indices, strength):
    p = np.zeros(n_regions, dtype=float)
    p[np.asarray(protected_indices, dtype=int)] = float(strength)
    return p


def simulate_protected_fk(times, y0, W, kappa, alpha, protected_indices, strength, mode):
    L = graph_laplacian(W)
    protection = make_protection_vector(len(y0), protected_indices, strength)

    sol = solve_ivp(
        protected_fk_rhs,
        t_span=(float(times[0]), float(times[-1])),
        y0=np.asarray(y0, dtype=float),
        t_eval=np.asarray(times, dtype=float),
        args=(L, float(kappa), float(alpha), protection, mode),
        method="RK45",
        rtol=1e-6,
        atol=1e-8,
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    return np.clip(sol.y.T, 0.0, 1.0)


def global_tau_auc(times, Y):
    global_tau = np.asarray(Y, dtype=float).sum(axis=1)
    duration = float(times[-1] - times[0])
    return float(np.trapezoid(global_tau, times) / duration)


def percent_reduction(unprotected_auc, protected_auc):
    return 100.0 * (unprotected_auc - protected_auc) / unprotected_auc


def run_one_protection_strategy(times, y0, W, kappa, alpha, strength, mode, protected_indices):
    unprotected = simulate_protected_fk(times, y0, W, kappa, alpha, [], 0.0, mode)
    protected = simulate_protected_fk(times, y0, W, kappa, alpha, protected_indices, strength, mode)
    return {
        "protected_indices": protected_indices,
        "unprotected_auc": global_tau_auc(times, unprotected),
        "protected_auc": global_tau_auc(times, protected),
        "percent_reduction": percent_reduction(
            global_tau_auc(times, unprotected),
            global_tau_auc(times, protected),
        ),
    }


## Plots

![Intracellular reduction boxplot](./plots/fk_protection/intra_top20_kappa_alpha.png)
![Extracellular reduction boxplot](./plots/fk_protection/extra_top20_kappa_alpha.png)


